In [1]:
import hopsworks

print("Hopsworks imported successfully!")

Hopsworks imported successfully!


In [2]:
import hopsworks

project = hopsworks.login(
    project="huzzproj10p",
    host="eu-west.cloud.hopsworks.ai"
)

print("Connected to project:", project.name)

2026-08-19 18:27:28,242 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai/p/42211
Connected to project: huzzproj10p


In [3]:
fs = project.get_feature_store()

print("Feature Store connected successfully!")

Feature Store connected successfully!


In [4]:
import pandas as pd

df = pd.read_csv("historical_aqi_features.csv")

print("Data loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df.head())

Data loaded successfully!
Rows: 2160
Columns: 23
                  time  pm2_5  pm10  carbon_monoxide  nitrogen_dioxide  \
0  2026-05-21 00:00:00   23.1  52.7            185.0              11.5   
1  2026-05-21 01:00:00   22.5  53.6            165.0               8.7   
2  2026-05-21 02:00:00   22.4  54.5            152.0               6.7   
3  2026-05-21 03:00:00   22.5  55.8            145.0               5.9   
4  2026-05-21 04:00:00   22.8  57.2            145.0               5.8   

   sulphur_dioxide  ozone  temperature_2m  relative_humidity_2m  \
0              5.0   56.0            28.7                    84   
1              4.5   59.0            28.3                    88   
2              4.2   60.0            28.2                    88   
3              4.2   60.0            28.2                    86   
4              4.4   59.0            28.3                    83   

   surface_pressure  ...  day  month  day_of_week  pm2_5_change  pm10_change  \
0            1003.1  ..

In [5]:
# Prepare historical data for Hopsworks

# Convert timestamp
df["time"] = pd.to_datetime(df["time"])

# Remove the first-row NaN values created by change calculations
df = df.dropna().reset_index(drop=True)

# Make sure numeric columns use numeric types
numeric_columns = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "precipitation",
    "hour",
    "day",
    "month",
    "day_of_week",
    "pm2_5_change",
    "pm10_change",
    "pm2_5_change_rate",
    "pm10_change_rate",
    "aqi",
    "aqi_change",
    "aqi_change_rate"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Remove any remaining invalid rows
df = df.dropna().reset_index(drop=True)

print("Data prepared successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing values:", df.isna().sum().sum())
print("Time range:", df["time"].min(), "to", df["time"].max())

Data prepared successfully!
Rows: 2159
Columns: 23
Missing values: 0
Time range: 2026-05-21 01:00:00 to 2026-08-18 23:00:00


In [6]:
# Create Hopsworks Feature Group

feature_group = fs.get_or_create_feature_group(
    name="aqi_historical_features",
    version=1,
    description="Historical hourly Karachi AQI features for 3-day AQI forecasting",
    primary_key=["time"],
    event_time="time",
    online_enabled=False
)

print("Feature Group created successfully!")
print("Name:", feature_group.name)
print("Version:", feature_group.version)

Feature Group created successfully!
Name: aqi_historical_features
Version: 1


In [7]:
# Upload historical AQI features to Hopsworks

feature_group.insert(
    df,
    write_options={"wait_for_job": True}
)

print("Historical data uploaded successfully!")
print("Rows uploaded:", len(df))

2026-08-19 18:32:21,349 WARNING: UserWarning: Casting timestamp column 'time' from 'ns' to 'us' will lose precision.



Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai/p/42211/fs/30895/fg/50886
Historical data uploaded successfully!
Rows uploaded: 2159


In [8]:
# Verify data in Hopsworks Feature Group

print("Checking Feature Group...")
print("Feature Group:", feature_group.name)
print("Version:", feature_group.version)

# Read the data back
stored_df = feature_group.read()

print("\nData retrieved successfully!")
print("Rows in Feature Store:", len(stored_df))
print("Columns:", len(stored_df.columns))

print("\nFirst 5 rows:")
display(stored_df.head())

Checking Feature Group...
Feature Group: aqi_historical_features
Version: 1


2026-08-19 18:32:55,903 ERROR: Set changed size during iteration. Detail: Failed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:10.240.28.16:5005 {created_time:"2026-08-19T18:32:55.903410655+00:00", grpc_status:2, grpc_message:"Set changed size during iteration. Detail: Failed"}. Client context: IOError: Server never sent a data message. Detail: Internal
Traceback (most recent call last):
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/hsfs/core/arrow_flight_client.py", line 433, in afs_error_handler_wrapper
    return func(instance, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/hsfs/core/arrow_flight_client.py", line 505, in _read_query
    return self._get_dataset(
           ^^^^^^^^^^^^^^^^^^
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/retrying.py", line 55, in wrapped_f
    return Retrying(*dargs, **dk

Error: Reading data from Hopsworks, using Hopsworks Feature Query Service           


FeatureStoreException: Could not read data using Hopsworks Query Service.

In [9]:
# Check Feature Group metadata

print("Feature Group:")
print("Name:", feature_group.name)
print("Version:", feature_group.version)
print("Description:", feature_group.description)

print("\nFeatures:")
for feature in feature_group.features:
    print(feature.name, "->", feature.type)

print("\nPrimary key:", feature_group.primary_key)
print("Event time:", feature_group.event_time)

2026-08-19 18:33:36,503 WARNING: HopsworksDeprecationWarning: hsfs.feature_group.FeatureGroupBase.features is deprecated. The function will be removed in a future release of hopsworks. Consider using hsfs.feature_group.FeatureGroupBase.columns instead.



Feature Group:
Name: aqi_historical_features
Version: 1
Description: Historical hourly Karachi AQI features for 3-day AQI forecasting

Features:
time -> timestamp
pm2_5 -> double
pm10 -> double
carbon_monoxide -> double
nitrogen_dioxide -> double
sulphur_dioxide -> double
ozone -> double
temperature_2m -> double
relative_humidity_2m -> bigint
surface_pressure -> double
wind_speed_10m -> double
precipitation -> double
hour -> bigint
day -> bigint
month -> bigint
day_of_week -> bigint
pm2_5_change -> double
pm10_change -> double
pm2_5_change_rate -> double
pm10_change_rate -> double
aqi -> bigint
aqi_change -> double
aqi_change_rate -> double

Primary key: ['time']
Event time: time


In [10]:
# Get the Feature Group
feature_group = fs.get_feature_group(
    name="aqi_historical_features",
    version=1
)

# Create a query using all features
query = feature_group.select_all()

# Read the training data
training_df = query.read()

print("Training data loaded from Hopsworks!")
print("Rows:", len(training_df))
print("Columns:", len(training_df.columns))
print(training_df.head())

2026-08-19 18:36:26,076 ERROR: Set changed size during iteration. Detail: Failed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:10.240.28.16:5005 {created_time:"2026-08-19T18:36:26.076174285+00:00", grpc_status:2, grpc_message:"Set changed size during iteration. Detail: Failed"}. Client context: IOError: Server never sent a data message. Detail: Internal
Traceback (most recent call last):
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/hsfs/core/arrow_flight_client.py", line 433, in afs_error_handler_wrapper
    return func(instance, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/hsfs/core/arrow_flight_client.py", line 505, in _read_query
    return self._get_dataset(
           ^^^^^^^^^^^^^^^^^^
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/retrying.py", line 55, in wrapped_f
    return Retrying(*dargs, **dk

Error: Reading data from Hopsworks, using Hopsworks Feature Query Service           


FeatureStoreException: Could not read data using Hopsworks Query Service.

In [11]:
# Read the Feature Group directly as Pandas
training_df = feature_group.read(
    dataframe_type="pandas",
    online=False
)

print("Training data loaded successfully!")
print("Rows:", len(training_df))
print("Columns:", len(training_df.columns))
print(training_df.head())

2026-08-19 18:37:00,949 ERROR: Set changed size during iteration. Detail: Failed. gRPC client debug context: UNKNOWN:Error received from peer ipv4:10.240.28.16:5005 {grpc_message:"Set changed size during iteration. Detail: Failed", grpc_status:2, created_time:"2026-08-19T18:37:00.949657786+00:00"}. Client context: IOError: Server never sent a data message. Detail: Internal
Traceback (most recent call last):
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/hsfs/core/arrow_flight_client.py", line 433, in afs_error_handler_wrapper
    return func(instance, *args, **kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/hsfs/core/arrow_flight_client.py", line 505, in _read_query
    return self._get_dataset(
           ^^^^^^^^^^^^^^^^^^
  File "/srv/hops/anaconda/envs/hopsworks_environment/lib/python3.12/site-packages/retrying.py", line 55, in wrapped_f
    return Retrying(*dargs, **dk

Error: Reading data from Hopsworks, using Hopsworks Feature Query Service           


FeatureStoreException: Could not read data using Hopsworks Query Service.

In [12]:
import pandas as pd

# Load uploaded historical AQI dataset
training_df = pd.read_csv("historical_aqi_features.csv")

print("Training data loaded successfully!")
print("Rows:", len(training_df))
print("Columns:", len(training_df.columns))
print("Missing values:", training_df.isnull().sum().sum())

training_df.head()


Training data loaded successfully!
Rows: 2160
Columns: 23
Missing values: 6


,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,surface_pressure,...,day,month,day_of_week,pm2_5_change,pm10_change,pm2_5_change_rate,pm10_change_rate,aqi,aqi_change,aqi_change_rate
0,2026-05-21 00:00:00,23.1,52.7,185.0,11.5,5.0,56.0,28.7,84,1003.1,...,21,5,3,NaN,NaN,NaN,NaN,77,NaN,NaN
1,2026-05-21 01:00:00,22.5,53.6,165.0,8.7,4.5,59.0,28.3,88,1002.7,...,21,5,3,-0.6,0.9,-2.597403,1.707780,76,-1.0,-1.298701
2,2026-05-21 02:00:00,22.4,54.5,152.0,6.7,4.2,60.0,28.2,88,1002.4,...,21,5,3,-0.1,0.9,-0.444444,1.679104,76,0.0,0.000000
3,2026-05-21 03:00:00,22.5,55.8,145.0,5.9,4.2,60.0,28.2,86,1002.3,...,21,5,3,0.1,1.3,0.446429,2.385321,76,0.0,0.000000
4,2026-05-21 04:00:00,22.8,57.2,145.0,5.8,4.4,59.0,28.3,83,1002.4,...,21,5,3,0.3,1.4,1.333333,2.508961,77,1.0,1.315789


In [13]:
# Prepare data for machine learning

# Remove rows containing NaN values created by change/rate calculations
model_df = training_df.dropna().copy()

print("ML dataset prepared!")
print("Rows:", len(model_df))
print("Columns:", len(model_df.columns))
print("Missing values:", model_df.isnull().sum().sum())

print("\nTime range:")
print("Start:", model_df["time"].min())
print("End:", model_df["time"].max())

ML dataset prepared!
Rows: 2159
Columns: 23
Missing values: 0

Time range:
Start: 2026-05-21 01:00:00
End: 2026-08-18 23:00:00


In [15]:
# Use the prepared dataset
ml_df = training_df.copy()

# Sort by time
ml_df = ml_df.sort_values("time").reset_index(drop=True)

# Use the first 80% for training and last 20% for testing
split_index = int(len(ml_df) * 0.8)

train_df = ml_df.iloc[:split_index].copy()
test_df = ml_df.iloc[split_index:].copy()

print("Train/Test split completed!")
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining period:")
print(train_df["time"].min(), "to", train_df["time"].max())

print("\nTesting period:")
print(test_df["time"].min(), "to", test_df["time"].max())

Train/Test split completed!
Training rows: 1728
Testing rows: 432

Training period:
2026-05-21 00:00:00 to 2026-07-31 23:00:00

Testing period:
2026-08-01 00:00:00 to 2026-08-18 23:00:00


In [16]:
# Create 72-hour (3-day) future AQI target

ml_df = ml_df.sort_values("time").reset_index(drop=True)

# AQI 72 hours in the future
ml_df["aqi_72h"] = ml_df["aqi"].shift(-72)

# Remove rows where the future AQI is unavailable
forecast_df = ml_df.dropna(subset=["aqi_72h"]).copy()

print("72-hour forecasting target created!")
print("Rows:", len(forecast_df))
print("Columns:", len(forecast_df.columns))

print("\nExample:")
print(
    forecast_df[["time", "aqi", "aqi_72h"]].head()
)

72-hour forecasting target created!
Rows: 2088
Columns: 24

Example:
                  time  aqi  aqi_72h
0  2026-05-21 00:00:00   77    105.0
1  2026-05-21 01:00:00   76    103.0
2  2026-05-21 02:00:00   76    100.0
3  2026-05-21 03:00:00   76     99.0
4  2026-05-21 04:00:00   77    101.0


In [17]:
# Prepare features and target for the ML model

# Features available at prediction time
feature_columns = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "precipitation",
    "hour",
    "day",
    "month",
    "day_of_week",
    "pm2_5_change",
    "pm10_change",
    "pm2_5_change_rate",
    "pm10_change_rate",
    "aqi",
    "aqi_change",
    "aqi_change_rate"
]

X = forecast_df[feature_columns]
y = forecast_df["aqi_72h"]

# Time-based split: first 80% training, last 20% testing
split_index = int(len(forecast_df) * 0.8)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("ML data prepared!")
print("Features:", len(feature_columns))
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nTraining target range:")
print(y_train.min(), "to", y_train.max())

print("\nTesting target range:")
print(y_test.min(), "to", y_test.max())

ML data prepared!
Features: 22
X_train: (1670, 22)
X_test: (418, 22)
y_train: (1670,)
y_test: (418,)

Training target range:
53.0 to 160.0

Testing target range:
56.0 to 107.0


In [18]:
# Train Random Forest AQI forecasting model

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Create the model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest model...")

# Train
model.fit(X_train, y_train)

print("Model training completed!")

# Make predictions
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation")
print("-------------------------")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

Training Random Forest model...
Model training completed!

Model Evaluation
-------------------------
MAE:  7.25
RMSE: 9.83
R²:   -0.3916


In [19]:
# Compare actual vs predicted AQI

results_df = test_df.copy()

# Align with the forecasting test set
results_df = forecast_df.iloc[split_index:].copy()

results_df["actual_aqi_72h"] = y_test.values
results_df["predicted_aqi_72h"] = y_pred

print("Prediction comparison:")
print(
    results_df[
        ["time", "aqi", "actual_aqi_72h", "predicted_aqi_72h"]
    ].head(20)
)

print("\nPrediction statistics:")
print(
    results_df[
        ["actual_aqi_72h", "predicted_aqi_72h"]
    ].describe()
)

Prediction comparison:
                     time  aqi  actual_aqi_72h  predicted_aqi_72h
1670  2026-07-29 14:00:00   85           107.0         107.729828
1671  2026-07-29 15:00:00   85            97.0          99.702452
1672  2026-07-29 16:00:00   85            93.0          95.845160
1673  2026-07-29 17:00:00   90            93.0          95.545356
1674  2026-07-29 18:00:00   89            90.0          90.714029
1675  2026-07-29 19:00:00   87            85.0          88.178378
1676  2026-07-29 20:00:00   83            81.0          86.760615
1677  2026-07-29 21:00:00   78            78.0          86.117087
1678  2026-07-29 22:00:00   76            77.0          84.794585
1679  2026-07-29 23:00:00   75            76.0          83.907312
1680  2026-07-30 00:00:00   74            73.0          80.121099
1681  2026-07-30 01:00:00   72            72.0          79.755801
1682  2026-07-30 02:00:00   69            69.0          79.586557
1683  2026-07-30 03:00:00   68            69.0       

In [20]:
# Baseline: predict future AQI using current AQI

baseline_pred = results_df["aqi"].values

baseline_mae = mean_absolute_error(
    results_df["actual_aqi_72h"],
    baseline_pred
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        results_df["actual_aqi_72h"],
        baseline_pred
    )
)

baseline_r2 = r2_score(
    results_df["actual_aqi_72h"],
    baseline_pred
)

print("Baseline Performance")
print("-------------------------")
print(f"MAE:  {baseline_mae:.2f}")
print(f"RMSE: {baseline_rmse:.2f}")
print(f"R²:   {baseline_r2:.4f}")

Baseline Performance
-------------------------
MAE:  6.62
RMSE: 9.15
R²:   -0.2081


In [21]:
# Check which features the Random Forest considered important

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Feature Importance:")
print(importance_df.to_string(index=False))

Feature Importance:
             feature  importance
                 day    0.278973
                hour    0.122618
      temperature_2m    0.121578
               ozone    0.071148
    surface_pressure    0.054187
                pm10    0.049046
         day_of_week    0.048840
      wind_speed_10m    0.035189
               pm2_5    0.032778
relative_humidity_2m    0.029469
     sulphur_dioxide    0.026837
               month    0.024072
     carbon_monoxide    0.023350
    nitrogen_dioxide    0.018836
                 aqi    0.016308
         pm10_change    0.014254
    pm10_change_rate    0.010003
   pm2_5_change_rate    0.007321
        pm2_5_change    0.006708
     aqi_change_rate    0.005387
          aqi_change    0.002072
       precipitation    0.001027


In [22]:
# Create time-series lag and rolling features

ts_df = ml_df.sort_values("time").copy()

# AQI lag features
ts_df["aqi_lag_1h"] = ts_df["aqi"].shift(1)
ts_df["aqi_lag_3h"] = ts_df["aqi"].shift(3)
ts_df["aqi_lag_6h"] = ts_df["aqi"].shift(6)
ts_df["aqi_lag_12h"] = ts_df["aqi"].shift(12)
ts_df["aqi_lag_24h"] = ts_df["aqi"].shift(24)
ts_df["aqi_lag_48h"] = ts_df["aqi"].shift(48)
ts_df["aqi_lag_72h"] = ts_df["aqi"].shift(72)

# Rolling AQI averages
ts_df["aqi_rolling_6h"] = ts_df["aqi"].rolling(6).mean()
ts_df["aqi_rolling_12h"] = ts_df["aqi"].rolling(12).mean()
ts_df["aqi_rolling_24h"] = ts_df["aqi"].rolling(24).mean()
ts_df["aqi_rolling_72h"] = ts_df["aqi"].rolling(72).mean()

# Future AQI target
ts_df["aqi_72h"] = ts_df["aqi"].shift(-72)

# Remove rows with unavailable lag/rolling/target values
ts_df = ts_df.dropna().reset_index(drop=True)

print("Time-series features created!")
print("Rows:", len(ts_df))
print("Columns:", len(ts_df.columns))

print("\nNew features:")
print([
    "aqi_lag_1h",
    "aqi_lag_3h",
    "aqi_lag_6h",
    "aqi_lag_12h",
    "aqi_lag_24h",
    "aqi_lag_48h",
    "aqi_lag_72h",
    "aqi_rolling_6h",
    "aqi_rolling_12h",
    "aqi_rolling_24h",
    "aqi_rolling_72h"
])

Time-series features created!
Rows: 2016
Columns: 35

New features:
['aqi_lag_1h', 'aqi_lag_3h', 'aqi_lag_6h', 'aqi_lag_12h', 'aqi_lag_24h', 'aqi_lag_48h', 'aqi_lag_72h', 'aqi_rolling_6h', 'aqi_rolling_12h', 'aqi_rolling_24h', 'aqi_rolling_72h']


In [23]:
# Prepare improved ML dataset

lag_features = [
    "aqi_lag_1h",
    "aqi_lag_3h",
    "aqi_lag_6h",
    "aqi_lag_12h",
    "aqi_lag_24h",
    "aqi_lag_48h",
    "aqi_lag_72h",
    "aqi_rolling_6h",
    "aqi_rolling_12h",
    "aqi_rolling_24h",
    "aqi_rolling_72h"
]

# Original weather and AQI features
base_features = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "precipitation",
    "hour",
    "month",
    "day_of_week"
]

# Combine features
improved_features = base_features + lag_features

X_improved = ts_df[improved_features]
y_improved = ts_df["aqi_72h"]

# Time-based 80/20 split
split_index_improved = int(len(ts_df) * 0.8)

X_train_improved = X_improved.iloc[:split_index_improved].copy()
X_test_improved = X_improved.iloc[split_index_improved:].copy()

y_train_improved = y_improved.iloc[:split_index_improved].copy()
y_test_improved = y_improved.iloc[split_index_improved:].copy()

print("Improved ML dataset prepared!")
print("Features:", len(improved_features))
print("Training rows:", len(X_train_improved))
print("Testing rows:", len(X_test_improved))

print("\nTraining period:")
print(ts_df.iloc[:split_index_improved]["time"].min())
print("to")
print(ts_df.iloc[:split_index_improved]["time"].max())

print("\nTesting period:")
print(ts_df.iloc[split_index_improved:]["time"].min())
print("to")
print(ts_df.iloc[split_index_improved:]["time"].max())

print("\nMissing values:")
print(X_train_improved.isna().sum().sum() + X_test_improved.isna().sum().sum())

Improved ML dataset prepared!
Features: 25
Training rows: 1612
Testing rows: 404

Training period:
2026-05-24 00:00:00
to
2026-07-30 03:00:00

Testing period:
2026-07-30 04:00:00
to
2026-08-15 23:00:00

Missing values:
0


In [24]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("Training improved Random Forest model...")

improved_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

improved_model.fit(X_train_improved, y_train_improved)

# Make predictions
y_pred_improved = improved_model.predict(X_test_improved)

# Evaluate
mae_improved = mean_absolute_error(y_test_improved, y_pred_improved)
rmse_improved = np.sqrt(mean_squared_error(y_test_improved, y_pred_improved))
r2_improved = r2_score(y_test_improved, y_pred_improved)

print("\nImproved Model Evaluation")
print("-------------------------")
print(f"MAE:  {mae_improved:.2f}")
print(f"RMSE: {rmse_improved:.2f}")
print(f"R²:   {r2_improved:.4f}")

Training improved Random Forest model...

Improved Model Evaluation
-------------------------
MAE:  5.97
RMSE: 7.78
R²:   0.0802


In [ ]:
# Feature importance for the improved Random Forest

importance_improved_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": improved_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("Improved Feature Importance:")
print(importance_improved_df.to_string(index=False))

ValueError: All arrays must be of the same length

In [27]:
print("Improved model features:", improved_model.n_features_in_)
print("X_train features:", X_train.shape[1])

Improved model features: 25
X_train features: 22


In [28]:
# Get the feature names used by the improved model

print("Improved model has", improved_model.n_features_in_, "features")

# Find the improved feature columns
improved_feature_columns = [
    col for col in improved_ml_df.columns
    if col not in ["time", "aqi_72h"]
]

print("Number of improved features:", len(improved_feature_columns))
print("Features:")
print(improved_feature_columns)

Improved model has 25 features


NameError: name 'improved_ml_df' is not defined

In [30]:
# Safely find all DataFrames in the notebook

dataframes = [
    (name, value.shape)
    for name, value in list(globals().items())
    if isinstance(value, pd.DataFrame)
]

print("DataFrames found:")
for name, shape in dataframes:
    print(name, "->", shape)

DataFrames found:
_ -> (5, 23)
df -> (2159, 23)
training_df -> (2160, 23)
_12 -> (5, 23)
model_df -> (2159, 23)
ml_df -> (2160, 24)
train_df -> (1728, 23)
test_df -> (432, 23)
forecast_df -> (2088, 24)
X -> (2088, 22)
X_train -> (1670, 22)
X_test -> (418, 22)
results_df -> (418, 26)
importance_df -> (22, 2)
ts_df -> (2016, 35)
X_improved -> (2016, 25)
X_train_improved -> (1612, 25)
X_test_improved -> (404, 25)


In [1]:
import requests
import pandas as pd

# Karachi coordinates
latitude = 24.8607
longitude = 67.0011

# Approximately 2 years of historical data
start_date = "2024-08-01"
end_date = "2026-08-18"

# Weather variables used by our AQI model
weather_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "precipitation"
]

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ",".join(weather_variables),
    "timezone": "Asia/Karachi",
    "wind_speed_unit": "kmh",
    "temperature_unit": "celsius",
    "precipitation_unit": "mm"
}

response = requests.get(url, params=params)

print("API status:", response.status_code)

response.raise_for_status()

weather_data = response.json()

weather_df = pd.DataFrame(weather_data["hourly"])

weather_df["time"] = pd.to_datetime(weather_df["time"])

print("\nHistorical weather data downloaded successfully!")
print("Rows:", len(weather_df))
print("Columns:", len(weather_df.columns))
print("\nTime range:")
print("Start:", weather_df["time"].min())
print("End:", weather_df["time"].max())

print("\nMissing values:")
print(weather_df.isnull().sum())

print("\nFirst 5 rows:")
display(weather_df.head())

API status: 200

Historical weather data downloaded successfully!
Rows: 17952
Columns: 6

Time range:
Start: 2024-08-01 00:00:00
End: 2026-08-18 23:00:00

Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
surface_pressure        0
wind_speed_10m          0
precipitation           0
dtype: int64

First 5 rows:


,time,temperature_2m,relative_humidity_2m,surface_pressure,wind_speed_10m,precipitation
0,2024-08-01 00:00:00,28.7,88,998.9,23.3,0.0
1,2024-08-01 01:00:00,28.6,88,998.3,22.4,0.0
2,2024-08-01 02:00:00,28.6,87,997.9,22.6,0.0
3,2024-08-01 03:00:00,28.7,85,997.6,23.9,0.0
4,2024-08-01 04:00:00,28.6,85,997.9,24.4,0.0


In [2]:
# Collect 2 years of historical air-quality data from Open-Meteo

air_quality_variables = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone"
]

aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

aq_params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ",".join(air_quality_variables),
    "timezone": "Asia/Karachi"
}

aq_response = requests.get(aq_url, params=aq_params)

print("API status:", aq_response.status_code)

aq_response.raise_for_status()

aq_data = aq_response.json()

aq_df = pd.DataFrame(aq_data["hourly"])

aq_df["time"] = pd.to_datetime(aq_df["time"])

print("\nHistorical air-quality data downloaded successfully!")
print("Rows:", len(aq_df))
print("Columns:", len(aq_df.columns))

print("\nTime range:")
print("Start:", aq_df["time"].min())
print("End:", aq_df["time"].max())

print("\nMissing values:")
print(aq_df.isnull().sum())

print("\nFirst 5 rows:")
display(aq_df.head())

ChunkedEncodingError: ('Connection broken: IncompleteRead(5700 bytes read, 4540 more expected)', IncompleteRead(5700 bytes read, 4540 more expected))

In [4]:
import pandas as pd
import requests

latitude = 24.8607
longitude = 67.0011
start_date = "2024-08-01"
end_date = "2026-08-18"

air_quality_variables = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
]

aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

# Break the range into yearly chunks to prevent server connection drops
date_ranges = [
    ("2024-08-01", "2025-07-31"),
    ("2025-08-01", "2026-08-18"),
]

dataframes = []

for start, end in date_ranges:
    print(f"Fetching air quality data from {start} to {end}...")

    aq_params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(air_quality_variables),
        "timezone": "Asia/Karachi",
    }

    aq_response = requests.get(aq_url, params=aq_params, timeout=60)
    aq_response.raise_for_status()

    aq_data = aq_response.json()
    chunk_df = pd.DataFrame(aq_data["hourly"])
    dataframes.append(chunk_df)

# Concatenate all chunks
aq_df = pd.concat(dataframes, ignore_index=True)
aq_df["time"] = pd.to_datetime(aq_df["time"])

print("\nAir quality data fetched successfully!")
print("Rows:", len(aq_df))

Fetching air quality data from 2024-08-01 to 2025-07-31...
Fetching air quality data from 2025-08-01 to 2026-08-18...

Air quality data fetched successfully!
Rows: 17952


In [6]:
import os
import numpy as np
import pandas as pd

# Ensure the 'data' directory exists
os.makedirs("data", exist_ok=True)

# 1. Merge Weather and Air Quality datasets on 'time'
df = pd.merge(aq_df, weather_df, on="time", how="inner")
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)


# 2. US EPA AQI Calculation Function for PM2.5
def calculate_pm25_aqi(pm25):
    if pd.isna(pm25):
        return np.nan
    breakpoints = [
        (0.0, 9.0, 0, 50),
        (9.1, 35.4, 51, 100),
        (35.5, 55.4, 101, 150),
        (55.5, 125.4, 151, 200),
        (125.5, 225.4, 201, 300),
        (225.5, 325.4, 301, 500),
    ]
    for c_low, c_high, i_low, i_high in breakpoints:
        if c_low <= pm25 <= c_high:
            return round(
                (i_high - i_low) / (c_high - c_low) * (pm25 - c_low) + i_low
            )
    return 500 if pm25 > 325.4 else np.nan


# 3. Compute AQI & Derivative Features
df["aqi"] = df["pm2_5"].apply(calculate_pm25_aqi)
df["pm2_5_change"] = df["pm2_5"].diff()
df["pm10_change"] = df["pm10"].diff()
df["pm2_5_change_rate"] = df["pm2_5"].pct_change() * 100
df["pm10_change_rate"] = df["pm10"].pct_change() * 100
df["aqi_change"] = df["aqi"].diff()
df["aqi_change_rate"] = df["aqi"].pct_change() * 100

# 4. Time-based Features
df["hour"] = df["time"].dt.hour
df["day"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["day_of_week"] = df["time"].dt.dayofweek

# 5. Time-Series Lag & Rolling Features
df["aqi_lag_1h"] = df["aqi"].shift(1)
df["aqi_lag_3h"] = df["aqi"].shift(3)
df["aqi_lag_6h"] = df["aqi"].shift(6)
df["aqi_lag_12h"] = df["aqi"].shift(12)
df["aqi_lag_24h"] = df["aqi"].shift(24)
df["aqi_lag_48h"] = df["aqi"].shift(48)
df["aqi_lag_72h"] = df["aqi"].shift(72)

df["aqi_rolling_6h"] = df["aqi"].rolling(6).mean()
df["aqi_rolling_12h"] = df["aqi"].rolling(12).mean()
df["aqi_rolling_24h"] = df["aqi"].rolling(24).mean()
df["aqi_rolling_72h"] = df["aqi"].rolling(72).mean()

# 6. Future 72-hour Target Column for Training
df["aqi_72h"] = df["aqi"].shift(-72)

# Save full 2-year combined raw/featured dataset locally
csv_path = "data/historical_aqi_weather_2years.csv"
df.to_csv(csv_path, index=False)

print("2-Year Master Dataset created successfully!")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"Time Range: {df['time'].min()} to {df['time'].max()}")
print(f"Backup saved to: {csv_path}")

2-Year Master Dataset created successfully!
Total Rows: 17952
Total Columns: 35
Time Range: 2024-08-01 00:00:00 to 2026-08-18 23:00:00
Backup saved to: data/historical_aqi_weather_2years.csv


In [1]:
%pip install hopsworks

  Using cached hopsworks-5.0.6-py3-none-any.whl.metadata (12 kB)
  Using cached pyhumps-1.6.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached furl-2.1.4-py2.py3-none-any.whl.metadata (25 kB)
  Using cached boto3-1.43.78-py3-none-any.whl.metadata (6.6 kB)
  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pyjks-20.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached mock-5.2.0-py3-none-any.whl.metadata (3.1 kB)
  Using cached avro-1.12.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached tzlocal-5.4.4-py3-none-any.whl.metadata (7.7 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached retrying-1.4.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached hopsworks_aiomysql-0.2.2-py3-none-any.whl.metadata (12 kB)
  Using 

  error: subprocess-exited-with-error
  
  × Building wheel for twofish (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [14 lines of output]
      C:\Users\checking\AppData\Local\Temp\pip-build-env-fg4cmx6e\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: BSD License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      error: Microsoft Visual C++ 14.0 or greater is required. Get it with "Microsoft C++ Build Tools": https://v

In [1]:
%pip install requests pandas

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.5.2-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl (9.8 MB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
Using cached numpy-2.5.2-cp313-cp313-win_amd64.whl (12.5 MB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   ---------------------------------------- 0/8 [urllib3]
   -------------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
%pip install hsfs

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "e:\10Pearls Shine Internship Project\.venv-1\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "e:\10Pearls Shine Internship Project\.venv-1\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "e:\10Pearls Shine Internship Project\.venv-1\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\checking\AppData\Local\Temp\pip-build-env-vgj_80_h\overlay\Lib\site-packages\setuptools\build_meta.p

In [1]:
import os
import hopsworks

# Set local certificate folder for Windows compatibility
os.environ["HOPSWORKS_CLIENT_CERT_FOLDER"] = "E:\\tmp"

print("Connecting to Hopsworks...")
project = hopsworks.login(
    project="huzzproj10p",
    host="eu-west.cloud.hopsworks.ai",
    api_key_value="nCmu6z9w6K90pWsf.FwfjvKKa4zp2gzqdbALSUZNxbsaqa9Knu9MeHvHnysIiJadKHzvVUaPKGBaDrCBC"
)
fs = project.get_feature_store()

print("Fetching Feature Group version 2...")
fg = fs.get_feature_group(name="aqi_historical_features", version=2)

query = fg.select_all()

print("Creating Feature View (aqi_forecast_feature_view)...")
feature_view = fs.get_or_create_feature_view(
    name="aqi_forecast_feature_view",
    version=1,
    description="Feature View for Karachi 3-Day AQI forecasting models",
    labels=["aqi"],
    query=query
)

print("\n==================================================")
print(" SUCCESS: Feature View Created & Registered!")
print("==================================================")

e:\10Pearls Shine Internship Project\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connecting to Hopsworks...


2026-08-24 22:28:50,575 INFO: Initializing external client
2026-08-24 22:28:50,576 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-24 22:29:00,923 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42211
Fetching Feature Group version 2...
Creating Feature View (aqi_forecast_feature_view)...
Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42211/fs/30895/fv/aqi_forecast_feature_view/version/1

 SUCCESS: Feature View Created & Registered!


In [6]:
# MY THIRD TRY TRAINING MODELS BOTH RANDOM FOREST AND RIDGE FOR COMPARISON
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import hopsworks

os.environ["HOPSWORKS_CLIENT_CERT_FOLDER"] = "E:\\tmp"

# 1. Connect to Hopsworks
project = hopsworks.login(
    project="huzzproj10p",
    host="eu-west.cloud.hopsworks.ai",
    api_key_value="nCmu6z9w6K90pWsf.FwfjvKKa4zp2gzqdbALSUZNxbsaqa9Knu9MeHvHnysIiJadKHzvVUaPKGBaDrCBC"
)
fs = project.get_feature_store()

# 2. Retrieve data directly from Feature Group to ensure target 'aqi' is present
fg = fs.get_feature_group(name="aqi_historical_features", version=2)
data = fg.select_all().read()

# Clean up column names in case Hopsworks attached prefixes
data.columns = [c.split(".")[-1] for c in data.columns]

# 3. Sort chronologically to prevent temporal data leakage
data['time'] = pd.to_datetime(data['time'])
data = data.sort_values('time').reset_index(drop=True)

# 4. Define features (X) and target (y)
target_col = 'aqi'
feature_cols = [c for c in data.columns if c not in [target_col, 'time']]

X = data[feature_cols].select_dtypes(include=[np.number])
y = data[target_col]

# 5. Chronological Train-Test Split (80% Train, 20% Test)
split_idx = int(len(data) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set size: {X_train.shape[0]} rows | Testing set size: {X_test.shape[0]} rows")
print(f"Features count: {X_train.shape[1]}")

# 6. Model 1: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# 7. Model 2: Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

# 8. Print Simultaneous Comparison Summary
print("\n==================================================")
print(" MODEL EVALUATION SUMMARY (2-YEAR HOPSWORKS DATA)")
print("==================================================")
print(f"Random Forest  -> MAE: {mae_rf:.2f} | RMSE: {rmse_rf:.2f} | R²: {r2_rf:.4f}")
print(f"Ridge Regression -> MAE: {mae_ridge:.2f} | RMSE: {rmse_ridge:.2f} | R²: {r2_ridge:.4f}")
print("==================================================")

2026-08-24 23:07:10,322 INFO: Closing external client and cleaning up certificates.
2026-08-24 23:07:10,326 INFO: Connection closed.
2026-08-24 23:07:10,329 INFO: Initializing external client
2026-08-24 23:07:10,330 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-24 23:07:12,646 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42211
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (5.55s) 
Training set size: 14304 rows | Testing set size: 3576 rows
Features count: 32

 MODEL EVALUATION SUMMARY (2-YEAR HOPSWORKS DATA)
Random Forest  -> MAE: 0.02 | RMSE: 0.33 | R²: 0.9996
Ridge Regression -> MAE: 0.00 | RMSE: 0.00 | R²: 1.0000


In [7]:
# MY FOURTH TRY: CORRECT 72-HOUR FORECASTING (PREVENTING DATA LEAKAGE)
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import hopsworks

os.environ["HOPSWORKS_CLIENT_CERT_FOLDER"] = "E:\\tmp"

# 1. Connect to Hopsworks and fetch the 2-year feature group
project = hopsworks.login(
    project="huzzproj10p",
    host="eu-west.cloud.hopsworks.ai",
    api_key_value="nCmu6z9w6K90pWsf.FwfjvKKa4zp2gzqdbALSUZNxbsaqa9Knu9MeHvHnysIiJadKHzvVUaPKGBaDrCBC"
)
fs = project.get_feature_store()
fg = fs.get_feature_group(name="aqi_historical_features", version=2)
data = fg.select_all().read()

# Standardize column names
data.columns = [c.split(".")[-1] for c in data.columns]

# 2. Sort chronologically by timestamp
data['time'] = pd.to_datetime(data['time'])
data = data.sort_values('time').reset_index(drop=True)

# 3. Create the 72-hour future target (shift target by -72 hours)
data['aqi_72h'] = data['aqi'].shift(-72)

# Drop the last 72 rows since their future target is unknown
data = data.dropna(subset=['aqi_72h']).reset_index(drop=True)

# 4. Define features (X) and target label (y)
target_col = 'aqi_72h'
feature_cols = [c for c in data.columns if c not in [target_col, 'time']]

X = data[feature_cols].select_dtypes(include=[np.number])
y = data[target_col]

# 5. Chronological Train-Test Split (80% Train, 20% Test)
split_idx = int(len(data) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Training set size: {X_train.shape[0]} rows | Testing set size: {X_test.shape[0]} rows")
print(f"Features count: {X_train.shape[1]}")

# 6. Model 1: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# 7. Model 2: Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

# 8. Print Results
print("\n==================================================")
print(" REAL 72-HOUR AQI FORECAST EVALUATION (2-YEAR DATA)")
print("==================================================")
print(f"Random Forest  -> MAE: {mae_rf:.2f} | RMSE: {rmse_rf:.2f} | R²: {r2_rf:.4f}")
print(f"Ridge Regression -> MAE: {mae_ridge:.2f} | RMSE: {rmse_ridge:.2f} | R²: {r2_ridge:.4f}")
print("==================================================")

2026-08-24 23:09:46,297 INFO: Closing external client and cleaning up certificates.
2026-08-24 23:09:46,300 INFO: Connection closed.
2026-08-24 23:09:46,302 INFO: Initializing external client
2026-08-24 23:09:46,303 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-24 23:09:48,489 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42211
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (8.40s) 
Training set size: 14246 rows | Testing set size: 3562 rows
Features count: 33

 REAL 72-HOUR AQI FORECAST EVALUATION (2-YEAR DATA)
Random Forest  -> MAE: 11.63 | RMSE: 16.34 | R²: -0.0285
Ridge Regression -> MAE: 11.45 | RMSE: 15.96 | R²: 0.0190


In [8]:
# MY FIFTH TRY: FEATURE DIAGNOSTICS & BASELINE COMPARISON
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 1. Print the list of 33 features used in training
print("==================================================")
print(" ACTIVE FEATURE LIST (33 FEATURES)")
print("==================================================")
for idx, col in enumerate(feature_cols, 1):
    print(f"{idx:02d}. {col}")

# 2. Evaluate Naïve Persistence Baseline (Predicting AQI(t+72) = AQI(t))
# Using 'aqi' column at prediction time as the guess for 'aqi_72h'
y_pred_baseline = X_test['aqi'] if 'aqi' in X_test.columns else X_test['aqi_lag_1h']

mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = root_mean_squared_error(y_test, y_pred_baseline)
r2_base = r2_score(y_test, y_pred_baseline)

print("\n==================================================")
print(" BASELINE VS MODEL COMPARISON")
print("==================================================")
print(f"Naïve Baseline   -> MAE: {mae_base:.2f} | RMSE: {rmse_base:.2f} | R²: {r2_base:.4f}")
print(f"Ridge Regression -> MAE: {mae_ridge:.2f} | RMSE: {rmse_ridge:.2f} | R²: {r2_ridge:.4f}")
print(f"Random Forest    -> MAE: {mae_rf:.2f} | RMSE: {rmse_rf:.2f} | R²: {r2_rf:.4f}")
print("==================================================")

 ACTIVE FEATURE LIST (33 FEATURES)
01. pm2_5
02. pm10
03. carbon_monoxide
04. nitrogen_dioxide
05. sulphur_dioxide
06. ozone
07. temperature_2m
08. relative_humidity_2m
09. surface_pressure
10. wind_speed_10m
11. precipitation
12. aqi
13. pm2_5_change
14. pm10_change
15. pm2_5_change_rate
16. pm10_change_rate
17. aqi_change
18. aqi_change_rate
19. hour
20. day
21. month
22. day_of_week
23. aqi_lag_1h
24. aqi_lag_3h
25. aqi_lag_6h
26. aqi_lag_12h
27. aqi_lag_24h
28. aqi_lag_48h
29. aqi_lag_72h
30. aqi_rolling_6h
31. aqi_rolling_12h
32. aqi_rolling_24h
33. aqi_rolling_72h

 BASELINE VS MODEL COMPARISON
Naïve Baseline   -> MAE: 13.79 | RMSE: 19.61 | R²: -0.4812
Ridge Regression -> MAE: 11.45 | RMSE: 15.96 | R²: 0.0190
Random Forest    -> MAE: 11.63 | RMSE: 16.34 | R²: -0.0285


In [ ]:
#REGISTER TOP MODEL TO HOPSWORKS MODEL REGISTRY
import os
import hopsworks

os.environ["HOPSWORKS_CLIENT_CERT_FOLDER"] = "E:\\tmp"

# 1. Connect to Hopsworks and access Model Registry
project = hopsworks.login(
    project="huzzproj10p",
    host="eu-west.cloud.hopsworks.ai",
    api_key_value="nCmu6z9w6K90pWsf.FwfjvKKa4zp2gzqdbALSUZNxbsaqa9Knu9MeHvHnysIiJadKHzvVUaPKGBaDrCBC"
)
mr = project.get_model_registry()

# 2. Save Ridge Regression model locally as artifact
import joblib
os.makedirs("model_dir", exist_ok=True)
joblib.dump(ridge, "model_dir/aqi_ridge_model.pkl")

# 3. Create Model Registry Entry with evaluation metrics
aqi_model = mr.python.create_model(
    name="karachi_aqi_72h_forecaster",
    metrics={
        "mae": float(mae_ridge),
        "rmse": float(rmse_ridge),
        "r2": float(r2_ridge)
    },
    description="Ridge Regression model predicting Karachi AQI 72 hours ahead using 2 years of weather and air-quality features."
)

# 4. Upload artifacts to cloud registry
aqi_model.save("model_dir")

print("\n==================================================")
print(" SUCCESS: Top Model Registered to Hopsworks Model Registry!")
print("==================================================")


2026-08-24 23:13:05,343 INFO: Closing external client and cleaning up certificates.
2026-08-24 23:13:05,347 INFO: Connection closed.
2026-08-24 23:13:05,349 INFO: Initializing external client
2026-08-24 23:13:05,350 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-24 23:13:07,113 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42211


Uploading model files (0 dirs, 0 files):  17%|█▋        | 1/6 [00:01<00:05,  1.04s/it]

Moving model files from 'model_dir' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading e:\10Pearls Shine Internship Project\Pearls AQI Predictor\notebook\model_dir/aqi_ridge_model.pkl: 100.000%|██████████| 1537/1537 elapsed<00:00 remaining<00:00
Model export complete: 100%|██████████| 6/6 [00:08<00:00,  1.42s/it]                   

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/42211/models/karachi_aqi_72h_forecaster/1

 SUCCESS: Top Model Registered to Hopsworks Model Registry!


In [10]:
import shap
import matplotlib.pyplot as plt

# 1. Initialize SHAP LinearExplainer for our trained Ridge Regression model
# We pass the fitted model (ridge) and the training feature background dataset (X_train)
explainer = shap.LinearExplainer(ridge, X_train)

# 2. Compute SHAP values for the test dataset (X_test)
shap_values = explainer(X_test)

# 3. Create a Global Summary Bar Plot of top feature importances
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("SHAP Global Feature Importance (72-Hour Karachi AQI Forecast)", fontsize=12)
plt.tight_layout()
plt.show()

# 4. Create a SHAP Beeswarm Plot (Shows feature value impact: High vs Low)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Beeswarm Plot: Feature Impact Direction on 72-Hour AQI", fontsize=12)
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'shap'